In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix, make_scorer
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# ============================================================================
# 1. DATA LOADING AND PREPROCESSING
# ============================================================================
def load_and_preprocess_data(train_path, test_path=None):
    """Load and preprocess embeddings data"""
    print("Loading training data...")
    with open(train_path, 'r') as f:
        train_data = json.load(f)
    
    X_train = []
    y_train = []
    for record in train_data:
        features = record['image_embedding'] + record['text_embedding']
        X_train.append(features)
        y_train.append(record['label'])
    
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    
    print(f"Training samples: {len(X_train)}")
    print(f"Feature dimension: {X_train.shape[1]}")
    print(f"Class distribution: 0={sum(y_train==0)}, 1={sum(y_train==1)}")
    print(f"Class imbalance ratio: {sum(y_train==0)/sum(y_train==1):.2f}:1")
    
    test_ids = None
    X_test = None
    if test_path:
        print("\nLoading test data...")
        with open(test_path, 'r') as f:
            test_data = json.load(f)
        
        X_test = []
        test_ids = []
        for record in test_data:
            features = record['image_embedding'] + record['text_embedding']
            X_test.append(features)
            test_ids.append(record['id'])
        
        X_test = np.array(X_test)
        print(f"Test samples: {len(X_test)}")
    
    return X_train, y_train, X_test, test_ids

# ============================================================================
# 2. ENHANCED LOGISTIC REGRESSION TUNING
# ============================================================================
def train_logistic_regression_extensive(X_train, y_train, X_val, y_val, search_type='grid'):
    """
    Extensive hyperparameter tuning for Logistic Regression
    search_type: 'grid' or 'random'
    """
    print("\n" + "="*80)
    print("EXTENSIVE LOGISTIC REGRESSION HYPERPARAMETER TUNING")
    print("="*80)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # Comprehensive parameter grid
    param_grid = {
        'C': [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0],
        'penalty': ['l1', 'l2', 'elasticnet'],
        'solver': ['liblinear', 'saga', 'lbfgs'],
        'max_iter': [500, 1000, 2000, 3000],
        'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}, {0: 1, 1: 5}],
        'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]  # For elasticnet
    }
    
    # Random search for faster exploration
    if search_type == 'random':
        param_distributions = {
            'C': np.logspace(-4, 2, 50),
            'penalty': ['l1', 'l2', 'elasticnet'],
            'solver': ['liblinear', 'saga', 'lbfgs'],
            'max_iter': [500, 1000, 2000, 3000],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}, {0: 1, 1: 5}],
            'l1_ratio': np.linspace(0.1, 0.9, 9)
        }
        
        lr = LogisticRegression(random_state=42)
        search = RandomizedSearchCV(
            lr, param_distributions,
            n_iter=200,  # Number of random combinations to try
            cv=5,
            scoring='f1_macro',
            n_jobs=-1,
            verbose=2,
            random_state=42
        )
    else:
        # Grid search with constraint handling
        print("Note: Using constrained grid search due to solver-penalty compatibility")
        
        # Create compatible parameter combinations
        lr = LogisticRegression(random_state=42)
        
        # Separate grids for different solvers
        param_grid_lbfgs = {
            'C': [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0],
            'penalty': ['l2', 'none'],
            'solver': ['lbfgs'],
            'max_iter': [500, 1000, 2000],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 5}]
        }
        
        param_grid_saga = {
            'C': [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0],
            'penalty': ['l1', 'l2', 'elasticnet'],
            'solver': ['saga'],
            'max_iter': [1000, 2000, 3000],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 5}],
            'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
        }
        
        param_grid_liblinear = {
            'C': [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear'],
            'max_iter': [500, 1000, 2000],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 5}]
        }
        
        # Run grid search for each solver
        best_score = 0
        best_params = None
        best_estimator = None
        
        for solver_name, pg in [('lbfgs', param_grid_lbfgs), 
                                 ('saga', param_grid_saga), 
                                 ('liblinear', param_grid_liblinear)]:
            print(f"\nSearching with {solver_name} solver...")
            
            search = GridSearchCV(
                lr, pg,
                cv=5,
                scoring='f1_macro',
                n_jobs=-1,
                verbose=1
            )
            
            search.fit(X_train_scaled, y_train)
            
            if search.best_score_ > best_score:
                best_score = search.best_score_
                best_params = search.best_params_
                best_estimator = search.best_estimator_
        
        # Create a mock search object for consistency
        class MockSearch:
            def __init__(self, best_params, best_score, best_estimator, cv_results):
                self.best_params_ = best_params
                self.best_score_ = best_score
                self.best_estimator_ = best_estimator
                self.cv_results_ = cv_results
        
        search = MockSearch(best_params, best_score, best_estimator, search.cv_results_)
    
    print(f"\n{'='*80}")
    print("LOGISTIC REGRESSION - BEST PARAMETERS")
    print(f"{'='*80}")
    for param, value in search.best_params_.items():
        print(f"  {param:20s}: {value}")
    print(f"\nBest CV F1 Score: {search.best_score_:.4f}")
    
    # Evaluate on validation set
    best_lr = search.best_estimator_
    y_val_pred = best_lr.predict(X_val_scaled)
    val_f1 = f1_score(y_val, y_val_pred, average='macro')
    
    print(f"\nValidation F1 Score: {val_f1:.4f}")
    print("\nValidation Classification Report:")
    print(classification_report(y_val, y_val_pred, target_names=['Not Important', 'Important']))
    
    # Print top 10 CV results for analysis
    print(f"\n{'='*80}")
    print("TOP 10 PARAMETER COMBINATIONS (by CV F1 Score)")
    print(f"{'='*80}")
    
    cv_results = pd.DataFrame(search.cv_results_)
    cv_results = cv_results.sort_values('mean_test_score', ascending=False).head(10)
    
    for idx, row in cv_results.iterrows():
        print(f"\nRank {cv_results.index.get_loc(idx) + 1}:")
        print(f"  Mean CV F1: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")
        params_dict = row['params']
        for k, v in params_dict.items():
            print(f"    {k}: {v}")
    
    # Generate next iteration parameters
    print(f"\n{'='*80}")
    print("SUGGESTED PARAMETERS FOR NEXT ITERATION")
    print(f"{'='*80}")
    
    best_C = search.best_params_['C']
    print(f"""
# Based on current best: C={best_C}
param_grid_next_iteration = {{
    'C': [{best_C * 0.5:.4f}, {best_C * 0.75:.4f}, {best_C:.4f}, {best_C * 1.25:.4f}, {best_C * 1.5:.4f}, {best_C * 2:.4f}],
    'penalty': ['{search.best_params_.get('penalty', 'l2')}'],
    'solver': ['{search.best_params_['solver']}'],
    'max_iter': [{search.best_params_['max_iter']}],
    'class_weight': {[search.best_params_['class_weight']]},
}}
""")
    
    return best_lr, scaler, val_f1, search.best_params_

# ============================================================================
# 3. ENHANCED GRADIENT BOOSTING TUNING
# ============================================================================
def train_gradient_boosting_extensive(X_train, y_train, X_val, y_val, search_type='grid'):
    """
    Extensive hyperparameter tuning for Gradient Boosting
    """
    print("\n" + "="*80)
    print("EXTENSIVE GRADIENT BOOSTING HYPERPARAMETER TUNING")
    print("="*80)
    
    n_neg = sum(y_train == 0)
    n_pos = sum(y_train == 1)
    scale_pos_weight = n_neg / n_pos
    print(f"Scale pos weight: {scale_pos_weight:.2f}")
    
    if search_type == 'random':
        # Random search for comprehensive exploration
        param_distributions = {
            'n_estimators': np.arange(50, 500, 50),
            'learning_rate': np.logspace(-3, 0, 20),
            'max_depth': np.arange(2, 12),
            'min_samples_split': np.arange(2, 21),
            'min_samples_leaf': np.arange(1, 11),
            'subsample': np.linspace(0.5, 1.0, 11),
            'max_features': ['sqrt', 'log2', None, 0.5, 0.7, 0.9],
            'loss': ['log_loss', 'exponential']
        }
        
        gb = GradientBoostingClassifier(random_state=42, verbose=0)
        
        search = RandomizedSearchCV(
            gb, param_distributions,
            n_iter=300,  # Try 300 random combinations
            cv=3,
            scoring='f1_macro',
            n_jobs=-1,
            verbose=2,
            random_state=42
        )
        
        print("Running randomized search (300 iterations)...")
        search.fit(X_train, y_train)
    
    else:
        # Multi-stage grid search
        print("\n--- STAGE 1: Coarse Grid Search ---")
        
        param_grid_stage1 = {
            'n_estimators': [50, 100, 200, 300, 400],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'max_depth': [3, 4, 5, 6, 7],
            'min_samples_split': [2, 5, 10, 15],
            'min_samples_leaf': [1, 2, 4, 6],
            'subsample': [0.7, 0.8, 0.9, 1.0],
            'max_features': ['sqrt', 'log2', None]
        }
        
        gb = GradientBoostingClassifier(random_state=42, verbose=0)
        
        search_stage1 = GridSearchCV(
            gb, param_grid_stage1,
            cv=3,
            scoring='f1_macro',
            n_jobs=-1,
            verbose=1
        )
        
        search_stage1.fit(X_train, y_train)
        
        print(f"\nStage 1 Best Score: {search_stage1.best_score_:.4f}")
        print("Stage 1 Best Parameters:")
        for param, value in search_stage1.best_params_.items():
            print(f"  {param}: {value}")
        
        # Stage 2: Fine-tuning around best parameters
        print("\n--- STAGE 2: Fine-Tuning Grid Search ---")
        
        best_p = search_stage1.best_params_
        
        param_grid_stage2 = {
            'n_estimators': [
                max(50, best_p['n_estimators'] - 50),
                best_p['n_estimators'],
                best_p['n_estimators'] + 50,
                best_p['n_estimators'] + 100
            ],
            'learning_rate': [
                best_p['learning_rate'] * 0.7,
                best_p['learning_rate'] * 0.85,
                best_p['learning_rate'],
                best_p['learning_rate'] * 1.15,
                best_p['learning_rate'] * 1.3
            ],
            'max_depth': [
                max(2, best_p['max_depth'] - 1),
                best_p['max_depth'],
                best_p['max_depth'] + 1
            ],
            'min_samples_split': [
                max(2, best_p['min_samples_split'] - 2),
                best_p['min_samples_split'],
                best_p['min_samples_split'] + 2
            ],
            'min_samples_leaf': [
                max(1, best_p['min_samples_leaf'] - 1),
                best_p['min_samples_leaf'],
                best_p['min_samples_leaf'] + 1
            ],
            'subsample': [
                max(0.5, best_p['subsample'] - 0.05),
                best_p['subsample'],
                min(1.0, best_p['subsample'] + 0.05)
            ],
            'max_features': [best_p['max_features']]
        }
        
        search = GridSearchCV(
            gb, param_grid_stage2,
            cv=5,  # More folds for fine-tuning
            scoring='f1_macro',
            n_jobs=-1,
            verbose=1
        )
        
        search.fit(X_train, y_train)
    
    print(f"\n{'='*80}")
    print("GRADIENT BOOSTING - BEST PARAMETERS")
    print(f"{'='*80}")
    for param, value in search.best_params_.items():
        print(f"  {param:20s}: {value}")
    print(f"\nBest CV F1 Score: {search.best_score_:.4f}")
    
    # Evaluate on validation set
    best_gb = search.best_estimator_
    y_val_pred = best_gb.predict(X_val)
    val_f1 = f1_score(y_val, y_val_pred, average='macro')
    
    print(f"\nValidation F1 Score: {val_f1:.4f}")
    print("\nValidation Classification Report:")
    print(classification_report(y_val, y_val_pred, target_names=['Not Important', 'Important']))
    
    # Feature importance
    print(f"\n{'='*80}")
    print("TOP 15 MOST IMPORTANT FEATURES")
    print(f"{'='*80}")
    feature_importance = best_gb.feature_importances_
    top_indices = np.argsort(feature_importance)[-15:][::-1]
    for rank, idx in enumerate(top_indices, 1):
        print(f"  {rank:2d}. Feature {idx:4d}: {feature_importance[idx]:.6f}")
    
    # Print top 10 CV results
    print(f"\n{'='*80}")
    print("TOP 10 PARAMETER COMBINATIONS (by CV F1 Score)")
    print(f"{'='*80}")
    
    cv_results = pd.DataFrame(search.cv_results_)
    cv_results = cv_results.sort_values('mean_test_score', ascending=False).head(10)
    
    for idx, row in cv_results.iterrows():
        print(f"\nRank {cv_results.index.get_loc(idx) + 1}:")
        print(f"  Mean CV F1: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")
        params_dict = row['params']
        for k, v in params_dict.items():
            print(f"    {k}: {v}")
    
    # Generate next iteration parameters
    print(f"\n{'='*80}")
    print("SUGGESTED PARAMETERS FOR NEXT ITERATION")
    print(f"{'='*80}")
    
    best_p = search.best_params_
    
    print(f"""
# Based on current best results
param_grid_next_iteration = {{
    'n_estimators': [{max(50, best_p['n_estimators'] - 50)}, {best_p['n_estimators']}, {best_p['n_estimators'] + 50}],
    'learning_rate': [{best_p['learning_rate'] * 0.8:.4f}, {best_p['learning_rate']:.4f}, {best_p['learning_rate'] * 1.2:.4f}],
    'max_depth': [{max(2, best_p['max_depth'] - 1)}, {best_p['max_depth']}, {best_p['max_depth'] + 1}],
    'min_samples_split': [{max(2, best_p['min_samples_split'] - 2)}, {best_p['min_samples_split']}, {best_p['min_samples_split'] + 2}],
    'min_samples_leaf': [{max(1, best_p['min_samples_leaf'] - 1)}, {best_p['min_samples_leaf']}, {best_p['min_samples_leaf'] + 1}],
    'subsample': [{max(0.5, best_p['subsample'] - 0.05):.2f}, {best_p['subsample']:.2f}, {min(1.0, best_p['subsample'] + 0.05):.2f}],
    'max_features': ['{best_p['max_features']}'],
}}
""")
    
    return best_gb, val_f1, search.best_params_

# ============================================================================
# 4. ENSEMBLE WITH OPTIMAL WEIGHTS
# ============================================================================
def train_ensemble_optimized(X_train, y_train, X_val, y_val, search_type='grid'):
    """Train ensemble with hyperparameter tuning for both models"""
    
    print("\n" + "="*80)
    print("TRAINING OPTIMIZED ENSEMBLE")
    print("="*80)
    
    # Train individual models with extensive tuning
    lr_model, scaler, lr_f1, lr_params = train_logistic_regression_extensive(
        X_train, y_train, X_val, y_val, search_type
    )
    
    gb_model, gb_f1, gb_params = train_gradient_boosting_extensive(
        X_train, y_train, X_val, y_val, search_type
    )
    
    # Test different ensemble weight combinations
    print("\n" + "="*80)
    print("OPTIMIZING ENSEMBLE WEIGHTS")
    print("="*80)
    
    X_val_scaled = scaler.transform(X_val)
    lr_probs = lr_model.predict_proba(X_val_scaled)
    gb_probs = gb_model.predict_proba(X_val)
    
    best_ensemble_f1 = 0
    best_weights = (0.5, 0.5)
    
    weight_combinations = [
        (w1/10, 1 - w1/10) for w1 in range(0, 11)
    ]
    
    print("\nTesting weight combinations:")
    for w_lr, w_gb in weight_combinations:
        ensemble_probs = w_lr * lr_probs + w_gb * gb_probs
        ensemble_pred = np.argmax(ensemble_probs, axis=1)
        f1 = f1_score(y_val, ensemble_pred, average='macro')
        
        print(f"  LR weight: {w_lr:.1f}, GB weight: {w_gb:.1f} → F1: {f1:.4f}")
        
        if f1 > best_ensemble_f1:
            best_ensemble_f1 = f1
            best_weights = (w_lr, w_gb)
    
    print(f"\nBest ensemble weights: LR={best_weights[0]:.1f}, GB={best_weights[1]:.1f}")
    print(f"Best ensemble F1: {best_ensemble_f1:.4f}")
    
    print(f"\n{'='*80}")
    print("ENSEMBLE SUMMARY")
    print(f"{'='*80}")
    print(f"Logistic Regression F1: {lr_f1:.4f}")
    print(f"Gradient Boosting F1:   {gb_f1:.4f}")
    print(f"Ensemble F1:            {best_ensemble_f1:.4f}")
    print(f"\nImprovement over best single model: {best_ensemble_f1 - max(lr_f1, gb_f1):.4f}")
    
    # Final predictions with best weights
    ensemble_probs = best_weights[0] * lr_probs + best_weights[1] * gb_probs
    ensemble_pred = np.argmax(ensemble_probs, axis=1)
    
    print("\nEnsemble Classification Report:")
    print(classification_report(y_val, ensemble_pred, target_names=['Not Important', 'Important']))
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_val, ensemble_pred)
    print(cm)
    
    # Save best parameters to file
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    best_params = {
        'timestamp': timestamp,
        'logistic_regression': lr_params,
        'gradient_boosting': gb_params,
        'ensemble_weights': {'lr': best_weights[0], 'gb': best_weights[1]},
        'performance': {
            'lr_f1': float(lr_f1),
            'gb_f1': float(gb_f1),
            'ensemble_f1': float(best_ensemble_f1)
        }
    }
    
    with open(f'best_params_{timestamp}.json', 'w') as f:
        json.dump(best_params, f, indent=2)
    
    print(f"\n✓ Best parameters saved to: best_params_{timestamp}.json")
    
    return lr_model, gb_model, scaler, best_weights, best_ensemble_f1

# ============================================================================
# 5. PREDICTION ON TEST SET
# ============================================================================
def predict_test_set(lr_model, gb_model, scaler, weights, X_test, test_ids):
    """Make predictions on test set using optimized ensemble"""
    
    print("\n" + "="*80)
    print("MAKING TEST PREDICTIONS")
    print("="*80)
    
    X_test_scaled = scaler.transform(X_test)
    
    lr_probs = lr_model.predict_proba(X_test_scaled)
    gb_probs = gb_model.predict_proba(X_test)
    
    ensemble_probs = weights[0] * lr_probs + weights[1] * gb_probs
    ensemble_pred = np.argmax(ensemble_probs, axis=1)
    
    print(f"Test predictions: 0={sum(ensemble_pred==0)}, 1={sum(ensemble_pred==1)}")
    print(f"Prediction distribution: {sum(ensemble_pred==1)/len(ensemble_pred)*100:.1f}% positive class")
    
    submission = pd.DataFrame({
        'row_id': test_ids,
        'target': ensemble_pred
    })
    
    return submission

# ============================================================================
# 6. MAIN PIPELINE
# ============================================================================
def main():
    """Main training and prediction pipeline"""
    
    print("="*80)
    print("ENHANCED HYPERPARAMETER TUNING PIPELINE")
    print("="*80)
    
    # Load data
    X_train, y_train, X_test, test_ids = load_and_preprocess_data(
        'train_part1.json',
        'test.json'
    )
    
    # Split training data
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train, y_train,
        test_size=0.2,
        random_state=42,
        stratify=y_train
    )
    
    # Choose search type: 'grid' or 'random'
    # 'random' is faster and explores more of the space
    # 'grid' is more thorough but slower
    search_type = 'grid'  # Change to 'random' for faster experimentation
    
    print(f"\nUsing {search_type.upper()} search strategy")
    
    # Train ensemble with extensive tuning
    lr_model, gb_model, scaler, best_weights, ensemble_f1 = train_ensemble_optimized(
        X_train_split, y_train_split,
        X_val_split, y_val_split,
        search_type=search_type
    )
    
    # Retrain on full training data with best parameters
    print("\n" + "="*80)
    print("RETRAINING ON FULL TRAINING DATA WITH BEST PARAMETERS")
    print("="*80)
    
    scaler_final = StandardScaler()
    X_train_scaled = scaler_final.fit_transform(X_train)
    
    # Use parameters from best models
    lr_final = LogisticRegression(**lr_model.get_params())
    lr_final.fit(X_train_scaled, y_train)
    print("✓ Logistic Regression trained")
    
    gb_final = GradientBoostingClassifier(**gb_model.get_params())
    gb_final.fit(X_train, y_train)
    print("✓ Gradient Boosting trained")
    
    # Predict on test set
    submission = predict_test_set(
        lr_final, gb_final, scaler_final,
        best_weights, X_test, test_ids
    )
    
    # Save submission
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f'logistic_v3.csv'
    submission.to_csv(filename, index=False)
    
    print(f"\n✓ Submission saved: {filename}")
    
    print("\n" + "="*80)
    print("PIPELINE COMPLETED")
    print("="*80)
    print(f"Expected performance: F1 > {ensemble_f1:.4f}")
    print("\nKey improvements:")
    print("  ✓ Extensive hyperparameter search")
    print("  ✓ Multi-stage grid search")
    print("  ✓ Optimized ensemble weights")
    print("  ✓ Best parameters saved for next iteration")
    print("  ✓ Top-10 parameter combinations analyzed")

if __name__ == "__main__":
    main()

ENHANCED HYPERPARAMETER TUNING PIPELINE
Loading training data...
Training samples: 1530
Feature dimension: 1024
Class distribution: 0=1326, 1=204
Class imbalance ratio: 6.50:1

Loading test data...
Test samples: 500

Using GRID search strategy

TRAINING OPTIMIZED ENSEMBLE

EXTENSIVE LOGISTIC REGRESSION HYPERPARAMETER TUNING
Note: Using constrained grid search due to solver-penalty compatibility

Searching with lbfgs solver...
Fitting 5 folds for each of 240 candidates, totalling 1200 fits

Searching with saga solver...
Fitting 5 folds for each of 1800 candidates, totalling 9000 fits


KeyboardInterrupt: 